# BÀI TẬP 1: HỒI QUY (Máy học – Ts. Nguyễn An Tế)
**Nội dung:** 1. Hồi quy đơn biến (GD, pseudo-inverse, sklearn) · 2. Chuẩn hóa · 3. IQ6 + VIF · 4. California Housing · 5. Car price · 6. Hồi quy đa thức · 7. OOP

> Dữ liệu ngoài (`IQ6.csv`, `Car_price_prediction.csv`) đặt trong thư mục `./Data`. **Nếu không có file, notebook tự sinh dữ liệu giả lập cùng cấu trúc** để code vẫn chạy được; thay file thật vào là có kết quả thật.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn as skl
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')
np.random.seed(42)

folder = './Data/'
os.makedirs(folder, exist_ok=True)

---
## 0. Lớp lõi `Linear_Regression` (câu 1.6 và 7)
Một lớp dùng chung cho đơn biến và đa biến, 3 cách giải:

| method | Ý tưởng | KHI NÀO DÙNG |
|---|---|---|
| `gd` | Lặp: θ ← θ − η·∇MSE | Dữ liệu rất lớn/nhiều features (không nghịch đảo ma trận được). **Bắt buộc chuẩn hóa** nếu features lệch thang đo, nếu không GD hội tụ chậm hoặc phân kỳ |
| `pinv` | θ = X⁺y (giả nghịch đảo) | Dữ liệu vừa/nhỏ, cần nghiệm chính xác; vẫn chạy khi XᵀX suy biến (đa cộng tuyến) |
| `sklearn` | `LinearRegression()` | Dùng làm chuẩn đối chiếu |

In [ ]:
class Linear_Regression:
    """Hồi quy tuyến tính (đơn/đa biến) với 3 phương pháp: gd, pinv, sklearn."""
    def __init__(self, method='pinv', lr=0.01, n_iter=10000, tol=1e-12):
        self.method, self.lr, self.n_iter, self.tol = method, lr, n_iter, tol
        self.theta_ = None; self.loss_ = []

    @staticmethod
    def _2d(X):
        X = np.asarray(X, dtype=float)
        return X.reshape(-1, 1) if X.ndim == 1 else X

    def fit(self, X, y):
        X = self._2d(X); y = np.asarray(y, dtype=float).ravel()
        if self.method == 'sklearn':
            m = LinearRegression().fit(X, y)
            self.theta_ = np.r_[m.intercept_, m.coef_]; return self
        Xb = np.c_[np.ones(len(X)), X]                  # thêm cột 1 cho hệ số chặn
        if self.method == 'pinv':
            self.theta_ = np.linalg.pinv(Xb) @ y
        elif self.method == 'gd':
            n = len(y); th = np.zeros(Xb.shape[1]); self.loss_ = []
            for _ in range(self.n_iter):
                err = Xb @ th - y
                th -= self.lr * (2 / n) * Xb.T @ err
                self.loss_.append(np.mean(err ** 2))
                if not np.isfinite(self.loss_[-1]): break   # phân kỳ
                if len(self.loss_) > 1 and abs(self.loss_[-2] - self.loss_[-1]) < self.tol: break
            self.theta_ = th
        else:
            raise ValueError('method phải là gd / pinv / sklearn')
        return self

    def predict(self, X):
        X = self._2d(X); return self.theta_[0] + X @ self.theta_[1:]

    @property
    def intercept_(self): return self.theta_[0]
    @property
    def coef_(self): return self.theta_[1:]

    @staticmethod
    def metrics(y, yhat):
        mse = mean_squared_error(y, yhat)
        return {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': mean_absolute_error(y, yhat), 'R2': r2_score(y, yhat)}


class Regression_Experiment:
    """Chạy cả 3 phương pháp trên cùng 1 bộ dữ liệu (có/không chuẩn hóa) và so sánh."""
    METHODS = ['gd', 'pinv', 'sklearn']
    def __init__(self, scaler=None, lr=0.01, n_iter=10000):
        self.scaler_name, self.lr, self.n_iter = scaler, lr, n_iter
        self.models = {}

    def _make_scaler(self):
        return {'standard': StandardScaler, 'robust': RobustScaler}.get(self.scaler_name, lambda: None)()

    def run(self, X_tr, X_te, y_tr, y_te):
        self.scaler = self._make_scaler()
        if self.scaler is not None:
            X_tr = self.scaler.fit_transform(X_tr); X_te = self.scaler.transform(X_te)
        rows = []
        for m in self.METHODS:
            mod = Linear_Regression(m, lr=self.lr, n_iter=self.n_iter).fit(X_tr, y_tr)
            self.models[m] = mod
            r = {'Mô hình': m, 'Intercept': mod.intercept_}
            for k, v in Linear_Regression.metrics(y_tr, mod.predict(X_tr)).items(): r[k + '_train'] = v
            for k, v in Linear_Regression.metrics(y_te, mod.predict(X_te)).items(): r[k + '_test'] = v
            rows.append(r)
        self.result = pd.DataFrame(rows).set_index('Mô hình')
        return self.result

---
## 1. Hồi quy tuyến tính ĐƠN BIẾN
### 1.1 Dữ liệu giả lập (100 quan sát, có nhiễu): y = 3x + 5 + ε

In [ ]:
n = 100
x = np.random.uniform(0, 10, n)
y = 3 * x + 5 + np.random.normal(0, 2, n)
x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=0.2, random_state=1)

plt.scatter(x, y, s=15); plt.xlabel('x'); plt.ylabel('y'); plt.title('Dữ liệu giả lập'); plt.show()

### 1.2 – 1.4 Xây dựng (GD, ma trận giả nghịch đảo, sklearn), đánh giá và so sánh

In [ ]:
exp1 = Regression_Experiment(scaler=None, lr=0.01, n_iter=20000)
res1 = exp1.run(x_tr.reshape(-1, 1), x_te.reshape(-1, 1), y_tr, y_te)
display(res1.round(4)) if 'display' in globals() else print(res1.round(4))

xs = np.linspace(0, 10, 100)
plt.scatter(x_tr, y_tr, s=15, label='train'); plt.scatter(x_te, y_te, s=15, c='r', label='test')
for m, mod in exp1.models.items():
    plt.plot(xs, mod.predict(xs), label=m, lw=2 if m == 'gd' else 1.5, ls='--' if m == 'sklearn' else '-')
plt.legend(); plt.title('Đường hồi quy của 3 mô hình'); plt.show()

hist = exp1.models['gd'].loss_
plt.plot(hist); plt.yscale('log'); plt.xlabel('Vòng lặp'); plt.ylabel('MSE'); plt.title('Hội tụ của Gradient Descent'); plt.show()

**Nhận xét:** ba mô hình cho hệ số gần như trùng nhau (hàm mục tiêu MSE lồi nên chỉ có một nghiệm tối ưu). `pinv` và `sklearn` cho nghiệm chính xác theo công thức; GD chỉ xấp xỉ nghiệm đó, độ lệch phụ thuộc `lr`, số vòng lặp và thang đo của x.

### 1.5 Nội suy và ngoại suy (lặp lại nhiều lần)
Huấn luyện trên x ∈ [0, 10]; **nội suy** = kiểm tra x ∈ [0, 10]; **ngoại suy** = kiểm tra x ∈ (10, 20].

In [ ]:
def thu_nghiem_noi_ngoai_suy(n_lan=50, n=100, sigma=2.0):
    rows = []
    for t in range(n_lan):
        rng = np.random.default_rng(t)
        xx = rng.uniform(0, 10, n); yy = 3 * xx + 5 + rng.normal(0, sigma, n)
        x_in = rng.uniform(0, 10, 50);  y_in = 3 * x_in + 5 + rng.normal(0, sigma, 50)
        x_out = rng.uniform(10, 20, 50); y_out = 3 * x_out + 5 + rng.normal(0, sigma, 50)
        for m in ['gd', 'pinv', 'sklearn']:
            mod = Linear_Regression(m, lr=0.01, n_iter=20000).fit(xx, yy)
            rows.append({'lan': t, 'mô hình': m,
                         'RMSE nội suy':  np.sqrt(mean_squared_error(y_in,  mod.predict(x_in))),
                         'RMSE ngoại suy': np.sqrt(mean_squared_error(y_out, mod.predict(x_out)))})
    return pd.DataFrame(rows)

kq = thu_nghiem_noi_ngoai_suy()
tom_tat = kq.groupby('mô hình')[['RMSE nội suy', 'RMSE ngoại suy']].agg(['mean', 'std']).round(4)
print(tom_tat)
kq.boxplot(column=['RMSE nội suy', 'RMSE ngoại suy'], by='mô hình', figsize=(9, 4)); plt.suptitle(''); plt.show()

**Nhận xét:** ngoại suy luôn kém ổn định hơn nội suy: sai số của hệ số góc bị khuếch đại khi đi xa miền dữ liệu huấn luyện, và nếu quan hệ thật không còn tuyến tính ngoài miền đó thì sai số còn lớn hơn nhiều. *Chỉ nên nội suy; ngoại suy phải cực kỳ thận trọng.*

---
## 2. Chuẩn hóa dữ liệu (StandardScaler, RobustScaler)
| Scaler | Công thức | KHI NÀO DÙNG |
|---|---|---|
| StandardScaler | (x−mean)/std | Dữ liệu gần phân phối chuẩn, ít ngoại lai |
| RobustScaler | (x−median)/IQR | Có ngoại lai (outliers) rõ rệt |

Chuẩn hóa không đổi chất lượng nghiệm của `pinv/sklearn`, nhưng giúp GD hội tụ nhanh hơn rất nhiều.

In [ ]:
for sc in ['standard', 'robust']:
    e = Regression_Experiment(scaler=sc, lr=0.1, n_iter=5000)
    r = e.run(x_tr.reshape(-1, 1), x_te.reshape(-1, 1), y_tr, y_te)
    print(f'=== {sc.upper()} ===  (số vòng GD thực tế: {len(e.models["gd"].loss_)})')
    print(r[['Intercept', 'RMSE_train', 'RMSE_test', 'R2_train', 'R2_test']].round(4), '\n')

---
## 3. IQ6.csv: Hồi quy tuyến tính ĐA BIẾN (target = IQ)
Hàm nạp dữ liệu: đọc `IQ6.csv`; nếu thiếu thì sinh dữ liệu mô phỏng (các môn có tương quan cao để minh họa đa cộng tuyến).

In [ ]:
def load_iq():
    p = folder + 'IQ6.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        return df.select_dtypes('number').dropna()
    rng = np.random.default_rng(0); n = 200
    toan = rng.normal(7, 1.2, n)
    ly   = 0.9 * toan + rng.normal(0, 0.35, n)          # Lý ~ Toán  → đa cộng tuyến
    hoa  = 0.8 * toan + rng.normal(0, 0.5, n)
    van  = rng.normal(6.5, 1.0, n)
    anh  = 0.5 * van + rng.normal(3, 0.8, n)
    iq   = 60 + 4 * toan + 2.5 * van + 1.5 * anh + rng.normal(0, 3, n)
    return pd.DataFrame({'Toan': toan, 'Ly': ly, 'Hoa': hoa, 'Van': van, 'Anh': anh, 'IQ': iq})

iq = load_iq(); target_iq = 'IQ' if 'IQ' in iq.columns else iq.columns[-1]
print(iq.shape, '| target:', target_iq); iq.head()

### 3.1 Mô hình đa biến (đủ features)

In [ ]:
def chay_da_bien(df, target, scaler='standard', lr=0.1, n_iter=5000, test_size=0.2, verbose=True):
    X = df.drop(columns=[target]); y = df[target]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=1)
    e = Regression_Experiment(scaler=scaler, lr=lr, n_iter=n_iter)
    r = e.run(Xtr, Xte, ytr, yte)
    if verbose: print(r[['RMSE_train', 'RMSE_test', 'R2_train', 'R2_test']].round(4))
    return e, r

print('Tất cả features:'); _ = chay_da_bien(iq, target_iq)

### 3.2 Variance Inflation Factor (VIF) xử lý đa cộng tuyến
VIFⱼ = 1/(1−R²ⱼ), với R²ⱼ là R² khi hồi quy feature j theo các feature còn lại.
**KHI NÀO LOẠI BIẾN:** VIF > 10 (chặt hơn: > 5) → feature gần như là tổ hợp tuyến tính của các feature khác; loại lần lượt biến có VIF lớn nhất rồi tính lại.

In [ ]:
class VIF_Selector:
    def __init__(self, nguong=10.0): self.nguong = nguong; self.lich_su = []

    @staticmethod
    def vif(X):
        X = pd.DataFrame(X); out = {}
        for c in X.columns:
            others = X.drop(columns=[c]).values; yj = X[c].values
            if others.shape[1] == 0: out[c] = 1.0; continue
            m = LinearRegression().fit(others, yj)
            r2 = r2_score(yj, m.predict(others))
            out[c] = np.inf if r2 >= 1 - 1e-12 else 1 / (1 - r2)
        return pd.Series(out)

    def fit(self, X):
        cols = list(X.columns)
        while len(cols) > 1:
            v = self.vif(X[cols]); self.lich_su.append(v.round(2).to_dict())
            if v.max() <= self.nguong: break
            cols.remove(v.idxmax())
        self.giu_lai_ = cols; return self

sel = VIF_Selector(10).fit(iq.drop(columns=[target_iq]))
print('VIF ban đầu:', sel.lich_su[0]); print('Features giữ lại:', sel.giu_lai_)
iq_vif = iq[sel.giu_lai_ + [target_iq]]
print('\nSau khi xử lý VIF:'); _ = chay_da_bien(iq_vif, target_iq)

### 3.3 Yêu cầu 1.2, 1.3, 1.4 trên IQ6 (GD / pinv / sklearn, đủ features vs sau VIF)

In [ ]:
for ten, d in [('Đủ features', iq), ('Sau VIF', iq_vif)]:
    e, r = chay_da_bien(d, target_iq, verbose=False)
    print(f'--- {ten} ---'); print(r[['RMSE_train', 'RMSE_test', 'R2_train', 'R2_test']].round(4))
    print('Hệ số (sklearn):', np.round(e.models['sklearn'].coef_, 3), '\n')

**Nhận xét:** VIF làm hệ số ổn định, dễ diễn giải hơn, còn R² gần như không giảm (thông tin bị loại vốn trùng lặp). Nếu R² giảm mạnh nghĩa là biến bị loại thật sự mang thông tin riêng, cần xem lại ngưỡng.

---
## 4. California Housing
### 4.1 Tải và lưu CSV (nếu không có mạng sẽ dùng dữ liệu mô phỏng cùng cấu trúc)

In [ ]:
def load_california():
    try:
        housing_data = skl.datasets.fetch_california_housing(as_frame=True)
        full_df = housing_data.frame               # toàn bộ features + target
        nguon = 'sklearn'
    except Exception as ex:
        rng = np.random.default_rng(1); n = 3000
        inc = rng.lognormal(1.3, 0.5, n); age = rng.uniform(1, 52, n)
        rooms = rng.normal(5.4, 1.2, n).clip(2); bed = rooms * 0.2 + rng.normal(0, .05, n)
        pop = rng.lognormal(7, .6, n); occ = rng.normal(3, .5, n).clip(1)
        lat = rng.uniform(32.5, 42, n); lon = -124 + (lat - 32.5) * -0.35 + rng.normal(0, .8, n) - 114 * 0 - 0.0
        val = 0.4 * inc + 0.005 * age + 0.05 * rooms - 0.1 * (lat - 36) + rng.normal(0, .4, n) + 1
        full_df = pd.DataFrame({'MedInc': inc, 'HouseAge': age, 'AveRooms': rooms, 'AveBedrms': bed,
                                'Population': pop, 'AveOccup': occ, 'Latitude': lat, 'Longitude': lon, 'MedHouseVal': val})
        nguon = 'mô phỏng (không tải được sklearn dataset)'
    full_df.to_csv(folder + 'California housing.csv', index=False)
    return full_df, nguon

cal, nguon = load_california(); print('Nguồn dữ liệu:', nguon, cal.shape); cal.head()

### 4.2 Thực hiện như câu 3 (đa biến → VIF → so sánh 3 mô hình)

In [ ]:
tg = 'MedHouseVal'
print('--- Đủ features ---'); e_all, r_all = chay_da_bien(cal, tg)
sel_c = VIF_Selector(10).fit(cal.drop(columns=[tg]))
print('\nVIF ban đầu:', sel_c.lich_su[0]); print('Giữ lại:', sel_c.giu_lai_)
print('--- Sau VIF ---'); e_vif, r_vif = chay_da_bien(cal[sel_c.giu_lai_ + [tg]], tg)

**Nhận xét:** `AveRooms`–`AveBedrms` và `Latitude`–`Longitude` thường có VIF cao (bản gốc California Housing). Loại biến làm mô hình đơn giản hơn nhưng R² có thể giảm vì Latitude/Longitude mang thông tin vị trí quan trọng; ở bài toán dự đoán thì cân nhắc giữ lại, còn bài toán diễn giải hệ số thì nên xử lý VIF.

---
## 5. Car_price_prediction (Kaggle)
### 5.1 Như câu 3

In [ ]:
def load_car():
    p = folder + 'Car_price_prediction.csv'
    if os.path.exists(p):
        df = pd.read_csv(p)
        if 'Mileage' in df: df['Mileage'] = pd.to_numeric(df['Mileage'].astype(str).str.replace(' km', '').str.replace(',', ''), errors='coerce')
        if 'Engine volume' in df: df['Engine volume'] = pd.to_numeric(df['Engine volume'].astype(str).str.replace(' Turbo', ''), errors='coerce')
        if 'Levy' in df: df['Levy'] = pd.to_numeric(df['Levy'].replace('-', 0), errors='coerce')
        df = df.rename(columns={'Prod. year': 'manufacture_year'})
        num = df.select_dtypes('number').drop(columns=[c for c in ['ID'] if c in df], errors='ignore')
        return num.dropna(), 'Price'
    rng = np.random.default_rng(2); n = 1500
    yr = rng.integers(1995, 2021, n); eng = rng.choice([1.2, 1.6, 2.0, 2.5, 3.5], n)
    mil = ((2021 - yr) * rng.normal(15000, 4000, n)).clip(1000)
    air = rng.integers(0, 16, n)
    mu = 9.0 + 0.07 * (yr - 2000) + 0.25 * eng - 4e-6 * mil + 0.02 * air
    price = np.exp(mu + rng.normal(0, 0.45, n))            # sai số nhân → phương sai tăng theo giá
    return pd.DataFrame({'manufacture_year': yr, 'Engine volume': eng, 'Mileage': mil, 'Airbags': air, 'Price': price}), 'Price'

car, tg_car = load_car(); print(car.shape); car.head()

In [ ]:
print('--- 5.1 Đa biến, đủ features ---'); e_car, r_car = chay_da_bien(car, tg_car)
sel_car = VIF_Selector(10).fit(car.drop(columns=[tg_car]))
print('\nVIF:', sel_car.lich_su[0], '→ giữ:', sel_car.giu_lai_)
print('--- Sau VIF ---'); _ = chay_da_bien(car[sel_car.giu_lai_ + [tg_car]], tg_car)

### 5.2 Biến mới `age_of_car = <năm hiện tại> − manufacture_year`

In [ ]:
from datetime import date
nam_hien_tai = date.today().year
car2 = car.copy()
car2['age_of_car'] = nam_hien_tai - car2['manufacture_year']
car2 = car2.drop(columns=['manufacture_year'])        # bỏ cột gốc vì tương quan hoàn hảo với age_of_car
print('--- 5.2 Có age_of_car ---'); e_car2, r_car2 = chay_da_bien(car2, tg_car)
# Ghi chú: age_of_car là phép biến đổi tuyến tính của manufacture_year nên R2 gần như không đổi;
# lợi ích là hệ số dễ diễn giải hơn (mỗi năm tuổi làm giá thay đổi bao nhiêu).

### 5.3 Residual plot (x = giá trị dự đoán, y = residual)
**Cách đọc:** điểm phân bố ngẫu nhiên quanh y = 0 → phương sai đồng nhất (homoscedasticity). Hình **phễu** → phương sai không đồng nhất (heteroscedasticity) → nên biến đổi target bằng log.

In [ ]:
def residual_plot(e, df, target, title, scaler_used=True, ax=None):
    X = df.drop(columns=[target]); y = df[target]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=1)
    Xte_s = e.scaler.transform(Xte) if e.scaler is not None else Xte
    yhat = e.models['sklearn'].predict(Xte_s)
    ax = ax or plt.gca()
    ax.scatter(yhat, yte - yhat, s=8, alpha=.5); ax.axhline(0, c='r')
    ax.set_xlabel('Giá trị dự đoán'); ax.set_ylabel('Residual'); ax.set_title(title)
    return yhat

plt.figure(figsize=(6, 4)); _ = residual_plot(e_car2, car2, tg_car, 'Residual plot – Price'); plt.show()

### 5.4 Biến đổi target bằng log(.) rồi so sánh

In [ ]:
car3 = car2.copy(); car3['log_Price'] = np.log(car3[tg_car]); car3 = car3.drop(columns=[tg_car])
print('--- 5.4 Target = log(Price) ---'); e_car3, r_car3 = chay_da_bien(car3, 'log_Price')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
residual_plot(e_car2, car2, tg_car, 'Trước: target = Price', ax=ax[0])
residual_plot(e_car3, car3, 'log_Price', 'Sau: target = log(Price)', ax=ax[1]); plt.tight_layout(); plt.show()

# So sánh công bằng trên cùng thang đo Price (đổi ngược bằng exp)
Xtr, Xte, ytr, yte = train_test_split(car3.drop(columns=['log_Price']), car3['log_Price'], test_size=0.2, random_state=1)
pred_price = np.exp(e_car3.models['sklearn'].predict(e_car3.scaler.transform(Xte)))
true_price = np.exp(yte)
print('Mô hình log, đổi ngược về Price → R2 test = %.4f | MAE = %.1f' % (r2_score(true_price, pred_price), mean_absolute_error(true_price, pred_price)))
print('Mô hình gốc, Price               → R2 test = %.4f' % r_car2.loc['sklearn', 'R2_test'])

**Nhận xét:** log(.) nén các giá lớn, làm residual đều hơn quanh 0 (bớt hình phễu) và quan hệ gần tuyến tính hơn. Khi so sánh, phải đổi ngược về cùng thang đo (hoặc so trên log) – không được so trực tiếp R² của hai target khác thang.

---
## 6. Hồi quy ĐA THỨC
### 6.1 Dữ liệu: y = 0.5x² + x + 2 + noise (100 quan sát)

In [ ]:
rng = np.random.default_rng(3)
xp = np.sort(rng.uniform(-3, 3, 100)); yp = 0.5 * xp ** 2 + xp + 2 + rng.normal(0, 0.5, 100)
xp_tr, xp_te, yp_tr, yp_te = train_test_split(xp.reshape(-1, 1), yp, test_size=0.3, random_state=1)
plt.scatter(xp, yp, s=15); plt.title('Dữ liệu bậc 2 + nhiễu'); plt.show()

### 6.2 Bậc 1, 2 và 15 (PolynomialFeatures)
**KHI NÀO:** bậc 1 → **underfit** (train và test đều sai số lớn); bậc đúng (2) → train ≈ test; bậc 15 → **overfit** (train rất tốt, test xấu, đường cong dao động).

In [ ]:
class Polynomial_Regression:
    def __init__(self, bac): self.bac = bac
    def fit(self, X, y):
        self.pf = PolynomialFeatures(self.bac, include_bias=False)
        Z = self.pf.fit_transform(X)
        self.sc = StandardScaler().fit(Z)                  # chuẩn hóa để ổn định số học (bậc cao)
        self.lr = LinearRegression().fit(self.sc.transform(Z), y); return self
    def predict(self, X): return self.lr.predict(self.sc.transform(self.pf.transform(X)))

rows = []; xs = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.figure(figsize=(14, 4))
for i, d in enumerate([1, 2, 15], 1):
    pm = Polynomial_Regression(d).fit(xp_tr, yp_tr)
    mtr = Linear_Regression.metrics(yp_tr, pm.predict(xp_tr)); mte = Linear_Regression.metrics(yp_te, pm.predict(xp_te))
    rows.append({'Bậc': d, 'RMSE_train': mtr['RMSE'], 'RMSE_test': mte['RMSE'], 'R2_train': mtr['R2'], 'R2_test': mte['R2']})
    plt.subplot(1, 3, i); plt.scatter(xp_tr, yp_tr, s=12, label='train'); plt.scatter(xp_te, yp_te, s=12, c='r', label='test')
    plt.plot(xs, pm.predict(xs), 'k'); plt.ylim(-2, 12); plt.title(f'Bậc {d}'); plt.legend()
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).set_index('Bậc').round(4))

**Phân tích:** bậc 1 underfit (bias cao, sai số train và test đều lớn). Bậc 2 khớp bản chất dữ liệu nên sai số test xấp xỉ độ nhiễu (σ = 0.5). Bậc 15 có RMSE_train thấp nhất nhưng RMSE_test **không cải thiện đáng kể** so với bậc 2 dù thêm 13 tham số, khoảng cách train–test lớn hơn, và đường cong dao động ở hai biên nơi ít dữ liệu (xem hình) → dấu hiệu overfit. Chọn độ phức tạp theo **sai số test/validation**, không theo sai số train. (Kết quả phụ thuộc hạt giống ngẫu nhiên; thử `random_state` khác hoặc bỏ chuẩn hóa để thấy overfit rõ hơn.)

---
## 7. Lập trình hướng đối tượng
Toàn bộ bài đã dùng OOP: `Linear_Regression` (mô hình), `Regression_Experiment` (so sánh/chuẩn hóa), `VIF_Selector` (đa cộng tuyến), `Polynomial_Regression` (đa thức). Các hàm `load_*` gom việc nạp dữ liệu.